# A detailed explanation of accuracy metrics — Part 1

## Context

The `Baseline_SARIMAX_LightGBM_Peak_VI.ipynb` notebook uses **two groups of metrics**, each answering a different question:

1. **Point-forecast metrics** — "how far is the forecast from the actual value?"  
   The metrics are **MAE, RMSE, WAPE, Bias, and MASE**.

2. **Peak-demand analysis** — "does the forecast support peak-capacity planning?"  
   The metrics are **breach recall/precision** and **peak magnitude and timing error**.

This part explains **group 1** (point forecast). Part 2 explains group 2.

**Notation:**
- $y_i$ = actual value at observation $i$
- $\hat y_i$ = forecast at observation $i$
- $n$ = number of observations
- $e_i = y_i - \hat y_i$ = error at observation $i$

---

## 1. MAE — Mean Absolute Error

**Formula:**

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat y_i|$$

**Meaning:** the average absolute error, in the same unit as the raw data (kW).  
If MAE = 1,500, the forecast is off by about 1,500 kW per hour on average, sometimes high and sometimes low.

**Why use it:**
- MAE treats every error equally (it does not amplify large errors the way RMSE does)
- Easy to explain in words
- It is the numerator of MASE (the comparison against a benchmark)

**Limitation:**
- It uses the same unit as the raw data, so two periods with different load levels (high summer vs low winter) cannot be compared directly
- To compare across periods, use WAPE or MASE

**Measured example (mean of the 3 backtest folds, Sep–Nov 2017):**

| Model | MAE |
|---|---|
| naive_1h (persistence) | 4.173 |
| naive_24h (seasonal-naive, benchmark) | 2.238 |
| LightGBM | 1.460 |
| SARIMAX | **1.122** |

**Interpretation:** SARIMAX has MAE ≈ 1,122, about 1,122 kW per hour on average, much lower than using the same hour yesterday (naive_24h, MAE ≈ 2,238).

---

## 2. RMSE — Root Mean Squared Error

**Formula:**

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat y_i)^2}$$

**Meaning:** like MAE (same unit as the raw data), but it **penalizes large errors** by squaring them before averaging.

Illustration:
- An hour off by 1,000 squares to 1,000,000
- An hour off by 10,000 squares to 100,000,000 (100 times larger, not 10 times)

So if MAE ≈ 2,000 but RMSE ≈ 5,000, a few hours have **very large errors**.

**Why use it:**
- Capacity planning cares about extreme errors, which usually happen at peak hours
- A high RMSE is a warning: the model looks fine on average but is **not safe for peak-related decisions**

**Limitation:**
- Harder to explain than MAE (it is not "off by X on average")
- Same limitation as MAE: it cannot be compared across periods with different load levels

**Measured example (mean of the 3 folds):**

| Model | MAE | RMSE | RMSE/MAE |
|---|---|---|---|
| naive_24h | 2.238 | 3.255 | 1,45 |
| LightGBM | 1.460 | 2.078 | 1,42 |
| SARIMAX | 1.122 | 1.514 | **1,35** |

**Interpretation:** SARIMAX has the lowest RMSE/MAE ratio (1.35), so its errors are not only smaller on average but also **more even** — fewer extreme hours than LightGBM and naive_24h.

---

## 3. WAPE — Weighted Absolute Percentage Error

**Formula:**

$$\text{WAPE} = \frac{\sum_{i=1}^{n} |y_i - \hat y_i|}{\sum_{i=1}^{n} |y_i|} \times 100\%$$

**Meaning:** the ratio of **total absolute error** to **total actual load**. It does **not depend on the unit or the scale** of the data.

WAPE = 2% means the total absolute error equals 2% of total actual load in that period.

**How it differs from MAPE:**
- **MAPE** divides each observation separately and then averages, so it **blows up** when $y_i \approx 0$
- **WAPE** divides a total by a total, so it stays **stable** even when some values are near 0

Tetouan load is never near 0, so MAPE would not explode, but WAPE is still preferred for the second reason below.

**Why use it:**
- **Scale-aware**, so periods with different load levels (summer vs winter) can be compared
- This is why the brief asks for WAPE as the main scale-aware metric
- Like MASE, WAPE does not depend on the unit or the scale

**Measured example (WAPE by fold, showing stability across folds):**

| Model | Fold 09 | Fold 10 | Fold 11 | Mean |
|---|---|---|---|---|
| naive_24h | 3,92% | 2,88% | 3,20% | 3,33% |
| LightGBM | 2,29% | 1,74% | 2,51% | 2,18% |
| SARIMAX | 1,57% | 1,64% | 1,83% | **1,68%** |

**Interpretation:** 
- SARIMAX stays around 1.6–1.8% in all three months (fold 09 is hot, fold 11 is cooler)
- LightGBM moves more: 1.74% in its best fold (10) and 2.51% in its worst fold (11)
- WAPE shows that SARIMAX is **more consistent across seasons**

---

## 4. Bias (mean error, signed)

**Formula:**

$$\text{Bias} = \frac{1}{n}\sum_{i=1}^{n} (\hat y_i - y_i)$$

**Meaning:** unlike MAE, RMSE, and WAPE, which use absolute values and are always positive, **bias keeps the sign** of the error.

- **Bias > 0**: the model tends to **forecast higher** than actual (over-forecast)
- **Bias < 0**: the model tends to **forecast lower** than actual (under-forecast)
- **Bias ≈ 0**: positive and negative errors cancel. That does not mean the forecast is accurate, only that it is not systematically high or low.

**Why it matters for capacity planning:**
- Bias has **asymmetric consequences**
- **Negative bias** (forecast too low) means the reserve is systematically short, which is a supply-risk
- **Positive bias** (forecast too high) means extra reserve and extra cost
- A good MAE, RMSE, or WAPE with negative bias still needs a decision adjustment, such as an extra safety margin

**Note:** MAE cannot detect this systematic bias. Two models can both have MAE = 1,000, while A is +1,000 every hour (Bias = +1,000) and B alternates between +1,500 and −500 (Bias ≈ 0). The MAE is the same, but the operational risk is not.

**Measured example (mean of the 3 folds):**

| Model | Bias |
|---|---|
| naive_24h | −38,79 |
| naive_168h | −911,23 |
| LightGBM | −125,49 |
| SARIMAX | **+30,78** |

**Interpretation:**
- **naive_168h** has a large negative bias (−911), so it forecasts clearly below actual and is not safe
- **LightGBM** has a mild negative bias (−125), so it leans slightly low and needs care
- **SARIMAX** has bias near 0 and slightly positive (+31), so it is more balanced and has no clear systematic bias

---

## 5. MASE — Mean Absolute Scaled Error

**Formula used in this notebook:**

$$\text{MASE} = \frac{\text{MAE}_{\text{model}}}{\text{MAE}_{\text{seasonal-naive 24h, on the same test set}}}$$

*(Note: this is a practical variant, not the original Hyndman & Koehler MASE. The original scales by the naive MAE on the training set. Here the seasonal-naive 24h MAE is computed on the **same test set**, so MASE says directly how many times better or worse the model is than the required benchmark.)*

**Meaning:**
- **MASE = 1.00**: the model is **exactly as accurate** as the seasonal-naive 24h benchmark
- **MASE < 1**: the model is **better** than the benchmark  
  - MASE = 0.5 means the error is **50%** of the benchmark error
  - MASE = 0.66 means the error is **66%** of the benchmark error
- **MASE > 1**: the model is **worse** than the required benchmark. That is a failure, because the benchmark is the simple rule "use yesterday's value".

**Why use it:**
- It answers the brief directly: "Does the model beat the required seasonal-naive baseline?"
- It does not depend on unit or scale, so folds can be compared
- It reports a **relative** value rather than an absolute one, which makes different contexts easier to compare

**Measured example (mean of the 3 folds):**

| Model | MASE |
|---|---|
| naive_1h | 1,90 |
| **naive_24h (benchmark, MASE = 1 by definition)** | **1.00** |
| naive_168h | 1,22 |
| naive_avg4w | 1,82 |
| **LightGBM** | **0,66** |
| **SARIMAX** | **0,51** |

**Interpretation:**
- Both LightGBM (0.66) and SARIMAX (0.51) **beat** the benchmark
- SARIMAX error is only **51%** of the benchmark error, about **twice as good**
- This meets the brief's "Baseline (Required)" item: a seasonal-naive baseline plus SARIMAX or LightGBM, and the model must be shown to beat the baseline

---

## Summary: which metric answers which question

| Operational question | Metric | Significance |
|---|---|---|
| "How many kW is the forecast off per hour, on average?" | MAE | Mean absolute error |
| "Are there extreme errors?" | RMSE (compared with MAE) | If RMSE is much larger than MAE, there are outlier errors |
| "Is this accuracy stable across periods with different load levels?" | WAPE | Scale-aware comparison across periods |
| "Does the model systematically forecast high or low?" | Bias | Detects over-forecast and under-forecast |
| "Is the model actually better than a simple rule, and by how much?" | MASE | Direct comparison with seasonal-naive 24h |

---

## Conclusion from the measured results

**From the 3 backtest folds (Sep–Nov 2017), SARIMAX:**

| Metric | Value | Interpretation |
|---|---|---|
| MAE | 1,122 | About 1,122 kW off per hour on average |
| RMSE | 1,514 | RMSE/MAE = 1.35 (errors are relatively even) |
| WAPE | 1.68% | Error is 1.68% of total load (stable across folds) |
| Bias | +31 | Slightly high, a small positive bias (safer) |
| MASE | 0.51 | Error is only 51% of the benchmark → a **clear win** |

→ **Conclusion:** SARIMAX not only beats the required baseline, it is about twice as good. The error is stable across seasons and there is no systematic bias, so it is a **good choice for the point forecast**.


# A detailed explanation of accuracy metrics — Part 2: Peak-Demand Analysis

## Context

This part explains **group 2**: metrics that judge the forecast from the point of view of **peak-capacity planning**, not only from the statistical error view in Part 1.

This group of metrics answers:
- "Does the forecast catch the hours that are actually **dangerous** (above the capacity threshold)?"
- "Does the model get the **timing and size** of the daily peak right?"
- "If a reserve margin is added, how does the trade-off between missed breaches and extra reserve change?"

---

## Part A: Breach detection

### A.1 Fixed capacity threshold

Before computing breach metrics, define **what counts as exceeding the threshold**.

The notebook uses the **P90 / P95 / P99 percentiles** of actual load, computed **once** on the earliest training period (Jan–Aug 2017, which includes the summer peak) and then **held fixed** for later periods.

This follows the brief's **fixed capacity threshold** requirement. The threshold is not updated with future data, which would be look-ahead leakage.

**Measured example (computed on Jan–Aug 2017, 5,663 hours):**

| Percentile | Threshold (kW) | Significance |
|---|---|---|
| P90 | 97,299 | 90% of hours have load at or below this threshold |
| P95 | 103,169 | 95% of hours have load at or below this threshold |
| P99 | 119,844 | 99% of hours have load at or below this threshold |

**Important note on sample size:** when these thresholds are applied to Sep–Dec 2017 (out of sample), very few hours breach them:
- **P90:** only 9 breach hours (out of 2,904 hours = 0.31%)
- **P95 and P99:** 0 breach hours

This is **not a model failure**. It is a real property of the data:
- The threshold is computed from summer, the highest peak of the year
- Sep–Dec is autumn and winter, when load is lower
- So the P95/P99 thresholds are too high for this period

→ **State this clearly in the report:** there are too few breach hours for a firm conclusion, and the bootstrap confidence interval for recall and precision will be very wide.

---

### A.2 Confusion matrix and metrics

For each hour, compare the **actual value** with the **threshold** and the **forecast** with the **threshold**:

```
                   Actual breach   |   Actual no breach
──────────────────────────────────────────────────────────────
Forecast breach        TP             |        FP (false alarm)
Forecast no breach  FN (missed)    |        TN
```

Definitions:
- **TP (true positive)**: actual breach **and** forecast breach → a **correct alert**
- **FP (false positive)**: no actual breach, but the forecast says breach → a **false alarm** (reserve is called when it is not needed)
- **FN (false negative)**: actual breach, but the forecast says no breach → a **miss** (the worst risk)
- **TN (true negative)**: no actual breach and no forecast breach → a normal hour

---

### A.3 Breach recall (sensitivity, true positive rate)

**Formula:**

$$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$

**Meaning:** of the hours that were **actually dangerous** (above the threshold), what **percentage** did the model alert on?

- **Recall = 100%** means no misses (every dangerous hour is caught)
- **Recall = 50%** means half of the dangerous hours are missed
- **Recall = 0%** means every dangerous hour is missed (useless)

**Why it matters:** a miss (FN) is the worst risk. The operator will not have capacity ready when the peak actually arrives, which can mean a shortfall.

**Limitation:** high recall does not mean the model is good if it raises **many false alarms** (high FP). Recall has to be traded off against precision.

**Measured example (P90 threshold, SARIMAX, Sep–Dec 2017):**

There are 9 hours that actually exceed P90, and the model forecasts a breach in all 9:
- TP = 9
- FN = 0
- **Recall = 9 / (9 + 0) = 100%**

Remember that this is only 9 observations. A 95% interval that sits at 100% is narrow, but it is not strong evidence because the sample is small.

---

### A.4 Breach precision (positive predictive value)

**Formula:**

$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$

**Meaning:** of the times the model **alerts** that the threshold will be exceeded, what percentage are **correct alerts**?

- **Precision = 100%** means every alert is correct (no false alarms)
- **Precision = 50%** means half of the alerts are correct and half are false
- **Precision = 0%** means every alert is false (useless, and it wastes reserve)

**Why it matters:** a false alarm (FP) calls reserve that is not needed, which costs money. Low precision is waste.

**Limitation:** high precision does not mean the model is good if it **misses many breaches** (high FN, low recall).

**Measured example (P90 threshold):**

The model forecasts a P90 breach in 20 hours:
- TP = 9 (correct alerts)
- FP = 11 (false alarms)
- **Precision = 9 / (9 + 11) = 9/20 = 45%**

Interpretation: only 45% of alerts are correct. Using them directly to call reserve is risky, because reserve would be called too often.

---

### A.5 Recall vs precision: the trade-off

These two metrics **always trade off against each other**:

| Strategy | Recall | Precision | results |
|---|---|---|---|
| **Alert very early** (low threshold, high margin) | High → few misses | Low → many false alarms | Safe, but expensive reserve |
| **Alert cautiously** (high threshold, low margin) | Low → many misses | High → few false alarms | Cheaper, but dangerous |

The notebook shows this trade-off with **reserve-margin scenarios**: add a reserve margin (0% to 20%) to the point forecast before comparing it with the threshold.

**How to choose a sensible alert level:**

Use the **newsvendor formula**:

$$\tau^* = \frac{C_u}{C_u + C_o}$$

where:
- $C_u$ = cost of **being short of capacity** (outage risk, penalties, compensation)
- $C_o$ = cost of **extra reserve** (standby plant, fuel)

**Example:**
- If $C_u : C_o = 2:1$ (a shortfall is **twice as costly** as extra reserve) then $\tau^* = 2/3 ≈ 67%$. Alert when the forecast P67 exceeds the threshold.
- If $C_u : C_o = 10:1$ (a shortfall is **ten times as costly**) then $\tau^* = 10/11 ≈ 91%$. Alert when the forecast P91 exceeds the threshold.

The notebook states this formula as a recommendation but does not apply it, because there are **no real cost figures** from the operator. Once those costs exist, $\tau^*$ can be computed and the matching reserve margin chosen.

---

### A.6 Bootstrap confidence intervals for recall and precision

Because breach hours are so few, especially at P95 and P99, a single recall or precision number is **not reliable**.

The notebook computes a **95% bootstrap confidence interval** by:
1. Split the out-of-sample data into **days** (blocks of 24 hours)
2. Repeat 1,000 times:
   - Resample **days** with replacement (do not resample individual hours, so the within-day structure stays intact)
   - Recompute recall and precision on that sample
3. Take the 2.5% and 97.5% percentiles of the 1,000 recall and precision values. That is the confidence interval.

**Example:** recall = 100% (1.00) with a 95% interval of [100%, 100%] does not mean "certainly 100%". It only means that all 1,000 resamples also returned 100%, and the whole result still rests on only 9 breach hours.

---

## Part B: Peak magnitude and timing errors

### B.1 Peak magnitude error

For **each day** in the evaluation period, compare:
- **Actual peak** = max(actual load during the day)
- **Forecast peak** = max(forecast during the day)

**Error formula:**

$$\text{magnitude\_error}_{\text{day}} = \text{max forecast} - \text{max actual}$$

The sign is kept. Positive means the **forecast peak is higher than actual** (over-forecast). Negative means the **forecast peak is lower** (under-forecast).

**Aggregated across days:**

| Metric | Formula | Significance |
|---|---|---|
| Peak magnitude MAE | $\frac{1}{\text{num\_days}}\sum \|\text{magnitude\_error}\|$ | Mean magnitude error (same unit, kW) |
| Peak magnitude RMSE | $\sqrt{\frac{1}{\text{num\_days}}\sum (\text{magnitude\_error})^2}$ | Penalizes days with a large peak error |
| Peak magnitude bias | $\frac{1}{\text{num\_days}}\sum \text{magnitude\_error}$ | Tendency to forecast the peak high or low |
| Peak magnitude MAPE (%) | $\frac{1}{\text{num\_days}}\sum \frac{\|\text{magnitude\_error}\|}{\text{max actual}} \times 100\%$ | **Relative** error against the size of the peak |

**Why it matters for capacity planning:**

The daily peak sets the **capacity that must be ready** for the next day. A large peak error means:
- **Positive bias** (peak forecast too high) is usually **safer** (better to prepare extra), but it costs more
- **Negative bias** (peak forecast too low) is **dangerous** (not enough capacity prepared)
- A **large error** in either direction means the plan cannot be trusted

**Measured example (SARIMAX, Sep–Dec 2017, 121 days):**

| Metric | Value | Interpretation |
|---|---|---|
| Peak magnitude MAE | 1,133 kW | About 1,133 kW off per day on average |
| Peak magnitude RMSE | 1,418 kW | A few days have a larger peak error than the MAE |
| Peak magnitude bias | +802 kW | The forecast peak is about **800 kW higher** than actual |
| Peak magnitude MAPE | 1.23% | The **relative** peak error is about 1% of the actual peak |

**Interpretation:** SARIMAX tends to forecast the daily peak a bit high (bias +800), but the error is about 1% of the peak. That is acceptable for planning, and a positive bias is safer than a negative one.

---

### B.2 Peak timing error

For **each day**, compare the **hour when the peak occurs**:

**Formula:**

$$\text{timing\_error}_{\text{day}} = \min(|h_{\text{forecast}} - h_{\text{actual}}|,\ 24 - |h_{\text{forecast}} - h_{\text{actual}}|)$$

The $\min(\cdot, 24-\cdot)$ term is a circular distance. The gap between 23:00 and 01:00 the next morning is 2 hours, not 22.

**Aggregated across days:**

| Metric | Formula | Significance |
|---|---|---|
| Mean timing error | $\frac{1}{\text{num\_days}}\sum \text{timing\_error}$ | Average number of hours off |
| Timing within ±1h | $\frac{\text{count}(\text{timing\_error} \le 1)}{\text{num\_days}}$ | Percent of days off by at most 1 hour |
| Timing within ±2h | $\frac{\text{count}(\text{timing\_error} \le 2)}{\text{num\_days}}$ | Percent of days off by at most 2 hours |

**Why it matters:**

Peak timing decides **when reserve capacity must be called**. A wrong time means reserve is called at the wrong time:
- Calling it too early means reserve runs for more hours than needed, which costs more
- Calling it too late means it is not ready in time

A **1-hour-ahead** model forecasts one hour at a time rather than the whole day at once. If each hour is accurate, the **chained forecast** can still follow the daily load curve, including the daily maximum of that chain.

**Measured example (SARIMAX, 121 days):**

| Metric | Value | Interpretation |
|---|---|---|
| Mean timing error | 0.28 hours | About 17 minutes off on average (0.28 × 60) |
| Within ±1h | 98% | **98% of days** are off by at most 1 hour |
| Within ±2h | 100% | Every day is off by at most 2 hours |

**Interpretation:** the 1-hour-ahead model tracks peak timing closely: 98% of days are within 1 hour and 100% within 2 hours. That is safe for deciding when to start reserve.

---

## Summary: which peak metric answers which question

| Operational question | Metric |
|---|---|
| "Of the hours that were actually dangerous, what percentage were alerted?" | Breach recall |
| "Of the alerts raised, what percentage were correct?" | Breach precision |
| "Which forecast percentile should trigger an alert, balancing misses and false alarms?" | Formula: τ* = Cu/(Cu+Co) |
| "If a 10% safety margin is added, how do misses and false alarms change?" | Reserve-margin matrix |
| "How far is the forecast daily peak from the actual peak?" | Peak magnitude MAE/RMSE/bias/MAPE |
| "Does the model get the peak hour right?" | Peak timing error |

---

## Metrics not yet used: pinball loss and empirical coverage (stretch section)

The brief also lists:
- **Pinball loss** = $\mathbb{E}[\max(\tau(y - q), (τ-1)(y-q))]$ for each quantile τ
- **Empirical coverage** = the share of observations that fall inside the forecast interval [q_low, q_high]

These two metrics are for a **probabilistic (quantile) forecast**. They belong to the **stretch/optional** section and are not implemented in this baseline. When quantile LightGBM is added later, they should be explained separately, because they score **an interval** rather than a single point forecast.